## Settings

In [167]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [ ]:
## libraries
import os
import random
import sys
from pathlib import Path

import numpy as np

## deterministic process settings
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.metrics import spec_marginal_delta
from src.evaluators.decomposing import (
    train_decomposed_separation,
    train_decomposed_attribution,
    compile_decomposed_separation,
    compile_decomposed_attribution,
    stat_decomposed_summary,
    stat_decomposed_attribution,
    stat_decomposed_test,
)

## constants
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET,
)

## Initialization

In [ ]:
## reproducibility
N_DECIMALS = 2
RANDOM_STATE = 42
N_REPEATS = 30
N_JOBS = 1

## deterministic runtime state
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
try:
    import torch

    torch.manual_seed(RANDOM_STATE)
    torch.set_num_threads(N_JOBS)
    torch.use_deterministic_algorithms(mode = True, warn_only = True)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(RANDOM_STATE)
        torch.cuda.manual_seed_all(RANDOM_STATE)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
except ImportError:
    pass

## load data and models
data = load_processed_data()
models = load_estimators(random_state = RANDOM_STATE, n_jobs = N_JOBS)

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

Original Data: 32 features, 25 observations
Learned Models: 9 estimators


## Training

In [ ]:
## train decomposed separation
cached_decomposed_separation = globals().get("results_dict_decomposed_separation")
if (
    not isinstance(cached_decomposed_separation, dict)
    or cached_decomposed_separation.get("n_repeats") != N_REPEATS
    or cached_decomposed_separation.get("random_state") != RANDOM_STATE
    or cached_decomposed_separation.get("n_jobs") != N_JOBS
):
    results_dict_decomposed_separation = train_decomposed_separation(
        data = data,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
        n_jobs = N_JOBS,
    )

Separation evaluation: 100%|██████████| 9/9 [05:41<00:00, 37.97s/model]


In [171]:
## train decomposed residual attribution
cached_decomposed_attribution = globals().get("results_dict_decomposed_attribution")
if (
    not isinstance(cached_decomposed_attribution, dict)
    or cached_decomposed_attribution.get("n_repeats") != N_REPEATS
    or cached_decomposed_attribution.get("random_state") != RANDOM_STATE
    or cached_decomposed_attribution.get("n_jobs") != N_JOBS
):
    results_dict_decomposed_attribution = train_decomposed_attribution(
        data = data,
        models = models,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
        n_jobs = N_JOBS,
    )

Residual attribution evaluation: 100%|██████████| 9/9 [02:04<00:00, 13.82s/model]


## Post-Processing

In [172]:
## compile decomposed separation results
results_decomposed_separation, predictions_decomposed_separation = compile_decomposed_separation(
    results = results_dict_decomposed_separation
)

## compile decomposed residual attribution results
results_decomposed_attribution, predictions_decomposed_attribution = compile_decomposed_attribution(
    results = results_dict_decomposed_attribution
)

## Residual Attribution Test
The residual attribution test asks whether, after estimating the capacity stage $C(X)$, the remaining slack is attributable primarily to dynamics rather than residual topological information. Under true additivity, $y^* = C(X) + R(Z)$, the slack $s = y^* - \hat{C}(X)$ should be more predictable from $Z$ than from $X$.

Within each LOGO (domain) fold, separate learners are fit for $X \to s$ and $Z \to s$, repeated across seeds, and their seed-averaged domain-level mean absolute errors are paired on model $\times$ domain.

- **$H_0$**: The residual slack is at least as predictable from $X$ as from $Z$ ($\Delta |\mathrm{error}| \le 0$).
- **$H_1$**: The residual slack is more predictable from $Z$ than from $X$ ($\Delta |\mathrm{error}| > 0$).

Rejecting $H_0$ supports that residual attribution lies with dynamics rather than topology after conditioning on $C(X)$.

### Empirical Test Specification
Repeated-seed-averaged predictions for $X \to s$ and $Z \to s$ are aggregated to model $\times$ domain by mean absolute error, then differenced as $\Delta = |\mathrm{error}|_{X \to s} - |\mathrm{error}|_{Z \to s}$. The paired differences are tested with a one-sided Wilcoxon signed-rank statistic ($H_1: \Delta > 0$). Effect size is reported as the rank-biserial correlation derived from the signed-rank statistic.

In [ ]:
## one-sided wilcoxon test for residual attribution
results_attribution, summary_attribution = stat_decomposed_attribution(
    predictions = predictions_decomposed_attribution,
    decimals = N_DECIMALS,
)

display(results_attribution)
display(summary_attribution)

Paired One-Sided Test (Wilcoxon Signed-Rank): n = 45
H0: Δ |ERROR| <= 0
H1: Δ |ERROR| > 0
Δ |ERROR|: paired X -> slack error minus Z -> slack error
Rank-biserial r: positive values favor Z -> slack
*** p < 0.001, ** p < 0.01, * p < 0.05


,N,Z BETTER,X BETTER,MEAN Δ |ERROR|,MEDIAN Δ |ERROR|,RANK-BISERIAL R,WILCOXON P,SIG.,DIFF.
0,45,32,13,268.34,0.65,0.52,0.0,**,Yes


,MEAN |ERROR|,STD |ERROR|,N
RESIDUAL FEATURES,,,
X -> SLACK,278.13,1633.82,45
Z -> SLACK,9.80,18.03,45


_The residual slack left behind by $\hat{C}(X)$ is better predicted from dynamics ($Z$) than from topology ($X$) in a majority of paired domain-level comparisons, with a positive median error shift and a significant one-sided Wilcoxon result. This supports a residual-attribution interpretation in which, after conditioning on $C(X)$, the remaining variation is more naturally assigned to dynamics than to unrecovered topological signal._

## Non-Inferiority Test
The additive specification is tested for non-inferiority against each non-additive alternative within an empirically derived EI margin $\delta$.

- **$H_0$**: The alternative specification exceeds additive by at least $\delta$ ($\Delta \mathrm{EI} \ge \delta$).
- **$H_1$**: The alternative specification does not exceed additive by $\delta$ ($\Delta \mathrm{EI} < \delta$).

Rejecting $H_0$ supports that the alternative specification fails to deliver an EI gain larger than $\delta$ over additive. In this setting, non-inferiority therefore means that the additive decomposition preserves EI transfer performance within the observed scale of additive-reference variability.

The sufficiency non-inferiority test evaluates whether the canonical additive specification $y^* = C(X) + R(Z)$ preserves EI transfer performance relative to richer and reduced alternatives. Each model is refit under additive, interaction, interaction-joint, joint, capacity-only, and dynamics-only specifications under LOGO-CV (domain), with predictions averaged across repeated seeds before scoring EI per (model $\times$ domain).

- **Additive**: $y^* = C(X) + R(Z)$ with separate capacity and residual stages.
- **Interaction**: Additive with $X \otimes Z$ cross-terms appended to the residual stage.
- **Interaction-Joint**: Single-stage learner over $[X, Z, X \otimes Z]$.
- **Joint**: Single-stage learner over $[X, Z]$ with no factorization.
- **Capacity-Only**: $y^* = C(X)$, dropping the residual stage.
- **Dynamics-Only**: $y^* = R(Z)$, dropping the capacity stage.

### Reporting Convention
Each specification contributes one repeated-seed-averaged value per (model $\times$ domain). The descriptive summary reports medians across (model $\times$ domain), with `EI` shown as median `[IQR]`; `VR`, `MV`, and `MS` are reported alongside as auxiliary frontier diagnostics. Higher `EI` indicates stronger transfer performance; lower `VR`, `MV`, and `MS` indicate fewer and smaller frontier violations. This descriptive table provides the empirical scale of the specification comparisons; the paired non-inferiority test evaluates whether any non-additive specification provides a practically meaningful EI improvement over the additive baseline.

### Empirical Test Specification
The non-inferiority margin $\delta$ is derived from variability across learners and held-out domains under the additive reference specification. Specifically, $\delta$ is set to the interquartile range of additive EI values, anchoring the margin to natural baseline performance dispersion rather than to alternative-specification effects. Paired gaps $\Delta \mathrm{EI} = \mathrm{EI}_{\text{spec}} - \mathrm{EI}_{\text{additive}}$ are tested against $\delta$ with a one-sided Wilcoxon signed-rank statistic. The table reports the median paired gap, rank-biserial effect size, one-sided p-value, Holm-adjusted p-value, significance code, and non-inferiority decision for each alternative specification.

In [174]:
## empirical delta specification
delta_sufficiency = spec_marginal_delta(
    results = results_decomposed_separation,
    feat_value = ["ei"],
    label_ref = "specification",
    value_ref = "additive",
    method = "iqr",
    scale = 1.0,  ## iqr range
    decimals = N_DECIMALS
)

## non-inferiority tests for decomposition sufficiency
results_table_decompose = stat_decomposed_test(
    results = results_decomposed_separation,
    delta = delta_sufficiency,
    decimals = N_DECIMALS,
)

display(results_table_decompose)

Paired Non-Inferiority Test (Wilcoxon Signed-Rank): n = 45, δ = 0.09
H₀: Δ EI ≥ δ
H₁: Δ EI < δ
Median Δ EI: Median of paired differences, not the difference of marginal medians
Rank-biserial r: Paired effect size, positive values favor non-inferiority
One-sided p: Wilcoxon signed-rank p-value for H₁
Holm-adj. p: Holm-Bonferroni adjusted one-sided p-value
NI.: Yes if Holm-adj. p < 0.05 and Median Δ EI < δ
Significance codes reflect Holm-adj. p
*** p < 0.001, ** p < 0.01, * p < 0.05


,Median Δ EI,Rank-biserial r,One-sided p,Holm-adj. p,Sig.,NI.
Specification,,,,,,
Interaction,0.00,0.91,0.00,0.00,***,Yes
Interaction Joint,-0.01,0.83,0.00,0.00,***,Yes
Joint,-0.01,0.83,0.00,0.00,***,Yes
Capacity Only,-0.01,0.98,0.00,0.00,***,Yes
Dynamics Only,-0.00,0.81,0.00,0.00,***,Yes


In [175]:
## median decomposition metrics by specification
decomposed_summary = stat_decomposed_summary(
    results = results_decomposed_separation,
    decimals = N_DECIMALS
)

display(decomposed_summary)

,EI [IQR],VR,MV,MS
SPECIFICATION,,,,
Additive,"0.78 [0.74, 0.82]",0.01,0.01,4.79
Interaction,"0.78 [0.73, 0.83]",0.00,0.00,5.27
Interaction Joint,"0.76 [0.74, 0.83]",0.00,0.00,5.60
Joint,"0.76 [0.74, 0.82]",0.00,0.00,4.86
Capacity Only,"0.76 [0.74, 0.80]",0.00,0.00,5.31
Dynamics Only,"0.77 [0.74, 0.83]",0.00,0.00,4.50


_The formal tests provide convergent support for the additive decomposition. The sufficiency non-inferiority test is the primary evidence for the separation claim: across relaxed alternatives, paired EI gaps are centered near zero, and Holm-adjusted Wilcoxon tests support non-inferiority within the empirically derived margin $\delta$. Thus, the richer interaction and joint specifications do not provide a practically meaningful EI gain over $C(X) + R(Z)$ under this evaluation. The residual attribution result complements this finding by showing that, after estimating $C(X)$, the remaining slack is better explained by dynamics ($Z$) than by topology ($X$). Together, these results favor $C(X) + R(Z)$ as a parsimonious decomposition that preserves transfer performance while separating structural capacity from residual dynamics._

## Visualization